# DSC LOGIKA UI 2026 — Credit Default Prediction (v76)

**Task.** Rank 30.000 test applications by the probability of reaching 90+ DPD
within 12 months of origination. Metric: Kolmogorov–Smirnov statistic.

**Model.** A rank-space blend of six gradient boosting legs and one spline
logistic regression, extrapolated along the ratio plateau found in v61:

```
trees4 = mean of rank(lgb_full, lgb_lean, cat_full, xgb_full)
core   = (2 * trees4 + lgbx_full) / 3
tm30   = 0.7 * core + 0.3 * linear
final  = rank01(tm30 + 0.5 * (tm30 - lgb_lean))
```

**What v76 is.** Not a new model. Eight scored submissions now show that the
public leaderboard is not one noisy cloud but two clusters that barely overlap,
and that inside this exact chain the leaderboard has a standard deviation of
**0.00007**. v76 therefore stops treating public as a lottery and starts using
it as a measuring instrument: a one-parameter-at-a-time sweep of the chain,
ordered so that the first submission tests whether the surface is smooth at
all. Section 6 has the measurement and the go/no-go rule.

**How to run.** `RUN_TRAINING = False` (default) rebuilds every candidate from
the stored fold predictions and verifies the reference one reproduces the
submitted CSV exactly; about two minutes. `RUN_TRAINING = True` rebuilds the
feature cache and retrains all legs; about 6.5 hours on 10 cores.

In [1]:
import hashlib, json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

RUN_TRAINING = False
RANDOM_STATE = 42

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data/train_applications.csv").exists() and (p / "features.py").exists():
            return p
    raise RuntimeError("repository root not found: run from inside the project")

ROOT = find_root(Path.cwd().resolve())
DATA = ROOT / "data"
V66, V76 = ROOT / "agas/v66", ROOT / "agas/v76"
os.chdir(ROOT)
(V76 / "output").mkdir(parents=True, exist_ok=True)

import sklearn
print("root        :", ROOT)
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("sklearn     :", sklearn.__version__)
print("RUN_TRAINING:", RUN_TRAINING)

root        : /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
numpy       : 2.5.1
pandas      : 3.0.5
sklearn     : 1.9.0
RUN_TRAINING: False


## 1. Data

Seven provided tables, nothing else. No external data is read in this notebook.

In [2]:
TABLES = ["train_applications", "test_applications", "customer_profiles",
          "previous_applications", "repayment_history", "account_behavior",
          "bureau_observations", "ecommerce_transactions"]
shapes = {}
for name in TABLES:
    n = sum(1 for _ in open(DATA / f"{name}.csv")) - 1
    cols = len(pd.read_csv(DATA / f"{name}.csv", nrows=1).columns)
    shapes[name] = (n, cols)

train = pd.read_csv(DATA / "train_applications.csv",
                    parse_dates=["application_datetime", "decision_datetime"])
test = pd.read_csv(DATA / "test_applications.csv",
                   parse_dates=["application_datetime", "decision_datetime"])

print(pd.DataFrame(shapes, index=["rows", "cols"]).T.to_string())
print(f"\ntrain {len(train):,} rows | test {len(test):,} rows "
      f"| positive rate {train.target.mean():.6f}")
print(f"customers in both sets: {len(set(train.customer_id) & set(test.customer_id))}")

                           rows  cols
train_applications        70000    10
test_applications         30000     9
customer_profiles        100000     8
previous_applications    201984    11
repayment_history       1361341     9
account_behavior        1084418    10
bureau_observations      797563    14
ecommerce_transactions  4359817    10

train 70,000 rows | test 30,000 rows | positive rate 0.135814
customers in both sets: 0


## 2. Cutoff audit

Every feature may use only information available at `decision_datetime` of the
selected application. Checked: whether any table holds a record dated after
that cutoff, and whether the bureau observation attached to the application was
available at decision time.

In [3]:
apps = pd.concat([train[["application_id", "customer_id", "decision_datetime"]],
                  test[["application_id", "customer_id", "decision_datetime"]]])
cut = apps[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"})
audit = {}

pa = pd.read_csv(DATA / "previous_applications.csv", parse_dates=["decision_datetime"]).merge(cut, on="customer_id")
audit["previous_applications.decision_datetime"] = (len(pa), int((pa.decision_datetime > pa.cut).sum()))
rh = pd.read_csv(DATA / "repayment_history.csv", parse_dates=["due_date", "last_payment_datetime"]).merge(cut, on="customer_id")
audit["repayment_history.due_date"] = (len(rh), int((rh.due_date > rh.cut).sum()))
audit["repayment_history.last_payment_datetime"] = (len(rh), int((rh.last_payment_datetime > rh.cut).sum()))
ab = pd.read_csv(DATA / "account_behavior.csv", parse_dates=["month"]).merge(cut, on="customer_id")
audit["account_behavior.month_closed"] = (len(ab), int(((ab.month + pd.offsets.MonthBegin(1)) > ab.cut).sum()))
ec = pd.read_csv(DATA / "ecommerce_transactions.csv", parse_dates=["transaction_datetime"]).merge(cut, on="customer_id")
audit["ecommerce_transactions.transaction_datetime"] = (len(ec), int((ec.transaction_datetime > ec.cut).sum()))
bu = pd.read_csv(DATA / "bureau_observations.csv", parse_dates=["report_datetime", "available_datetime"])
sel = bu.merge(apps, on=["application_id", "customer_id"], how="inner")
audit["bureau_observations.available_datetime"] = (len(sel), int((sel.available_datetime > sel.decision_datetime).sum()))

rows = pd.DataFrame([(k, v[0], v[1]) for k, v in audit.items()],
                    columns=["source field", "rows joined", "rows after cutoff"])
print(rows.to_string(index=False))
post = int(rows["rows after cutoff"].sum())
print(f"\ntotal records dated after the cutoff: {post}")
assert post == 0, "a source holds post-cutoff records; window features need a positive-time filter"
print("PASS: the provided tables are already truncated at decision_datetime")
del pa, rh, ab, ec, bu, sel

                               source field  rows joined  rows after cutoff
    previous_applications.decision_datetime       201984                  0
                 repayment_history.due_date      1361341                  0
    repayment_history.last_payment_datetime      1361341                  0
              account_behavior.month_closed      1084418                  0
ecommerce_transactions.transaction_datetime      4359817                  0
     bureau_observations.available_datetime       264088                  0

total records dated after the cutoff: 0
PASS: the provided tables are already truncated at decision_datetime


## 3. Features

`features.py` builds one row per application from the seven tables. Every
historical aggregate is computed as `days_ago = decision_datetime - event` and
windowed over that quantity. Target-encoded factor features are recomputed
inside each fold from training fold labels only. Column sets: `full` 598,
`lean` 490, `fac` 553.

v76 does not touch `features.py` and trains nothing. v75 measured that this
feature family is exhausted: a learning curve that buys only +0.0007 AUC from
the last 20 percent of rows, monotone constraints at −0.00078, and compact and
additive models all below the 490-column baseline.

In [4]:
CACHE = ROOT / "agas/shared/cache/features_cache.pkl"
EXTRA = ROOT / "agas/shared/cache/extra_cache.pkl"

if RUN_TRAINING:
    import features as ft
    t0 = time.time()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    ft.build().to_pickle(CACHE)
    rc = os.system("python3 agas/shared/build_extra.py")
    assert rc == 0, f"build_extra.py failed with code {rc}"
    print(f"feature cache rebuilt in {(time.time() - t0) / 60:.1f} min")

NOISE_EXACT = {"region", "employment_type", "housing_status", "age",
               "employment_tenure_months", "dependents", "tenure_to_age",
               "income_per_dependent", "loan_purpose", "decision_lag_h",
               "app_hour", "app_dow"}
NOISE_PREFIX = ("ec_share_", "ec_catqty_", "cs_shift", "buh_", "gn_")

if CACHE.exists() and EXTRA.exists():
    f = pd.read_pickle(CACHE)
    ex = pd.read_pickle(EXTRA)
    add = [c for c in ex.columns if c != "application_id" and c not in f.columns]
    f = f.merge(ex[["application_id"] + add], on="application_id", how="left")
    cols = [c for c in f.columns if c not in ("application_id", "customer_id", "target")]
    lean = [c for c in cols if c not in NOISE_EXACT and not c.startswith(NOISE_PREFIX)]
    print(f"feature frame : {f.shape[0]:,} rows x {len(cols)} columns")
    print(f"full / lean   : {len(cols)} / {len(lean)}")
    del f, ex
else:
    print("feature cache not present; set RUN_TRAINING = True to rebuild it")

feature frame : 100,000 rows x 598 columns
full / lean   : 598 / 490


## 4. Models

Seven legs, 10-fold stratified splits repeated over six fold partitions (seeds
42, 7, 13, 21, 5, 101), LightGBM legs also averaged over five booster seeds.
Nothing is fitted on test.

| leg | model | columns |
|---|---|---|
| `lgb_full` | LightGBM, 15 leaves, lr 0.01 | full |
| `lgb_lean` | same parameters | lean |
| `lgbx_full` | LightGBM, 31 leaves, lr 0.012 | full |
| `cat_full` | CatBoost | full |
| `xgb_full` | XGBoost | full |
| `fac_full` | LightGBM on fold-safe factor features | fac + factor |
| `linear` | spline logistic regression | lean |

In [5]:
TRAIN_CMD = ("python3 agas/v66/training/train_submission_v66.py "
             "--folds 10 --fold-seeds 42,7,13,21,5,101 --lgb-seeds 0,1,2,3,4 "
             "--lr 0.01 --l2 5.0 --out-prefix v66")
V76_CMD = "python3 agas/v76/training/make_submission_v76.py"
if RUN_TRAINING:
    for c in (TRAIN_CMD, V76_CMD):
        rc = os.system(c)
        assert rc == 0, f"{c} failed with code {rc}"
else:
    print("training skipped. the commands behind the stored predictions were:")
    for c in (TRAIN_CMD, V76_CMD):
        print(" ", c)

training skipped. the commands behind the stored predictions were:
  python3 agas/v66/training/train_submission_v66.py --folds 10 --fold-seeds 42,7,13,21,5,101 --lgb-seeds 0,1,2,3,4 --lr 0.01 --l2 5.0 --out-prefix v66
  python3 agas/v76/training/make_submission_v76.py


## 5. The chain, rebuilt and verified

The reference vector is rebuilt from the stored predictions and checked against
the submitted CSV: ranks must match to floating point and Spearman must be 1.0.

In [6]:
def rank01(v):
    return rankdata(v) / len(v)

def ks_stat(score, y):
    order = np.argsort(score, kind="mergesort")
    yy = y[order]
    return float(np.abs(np.cumsum(yy) / yy.sum()
                        - np.cumsum(1 - yy) / (1 - yy).sum()).max())

z = np.load(V66 / "output/preds_v66.npz", allow_pickle=True)
train_ids, test_ids = z["train_ids"], z["test_ids"]
y = train.set_index("application_id").loc[train_ids].target.to_numpy().astype(int)
ro = {k[4:]: rank01(z[k]) for k in z.files if k.startswith("oof_")}
rt = {k[5:]: rank01(z[k]) for k in z.files if k.startswith("test_")}

def chain(d, tm=0.30, r=0.50, ref="lgb_lean", cw=2.0):
    trees4 = (d["lgb_full"] + d["lgb_lean"] + d["cat_full"] + d["xgb_full"]) / 4
    core = (trees4 * cw + d["lgbx_full"]) / (cw + 1.0)
    base = (1 - tm) * core + tm * d["linear"]
    return rank01(base + r * (base - d[ref]))

oof_final, test_final = chain(ro), chain(rt)

print("leg performance, out of fold")
print(f"{'leg':12s} {'AUC':>9} {'KS':>9}")
for k in sorted(ro, key=lambda k: -roc_auc_score(y, ro[k])):
    print(f"{k:12s} {roc_auc_score(y, ro[k]):9.6f} {ks_stat(ro[k], y):9.6f}")
print(f"{'CHAIN':12s} {roc_auc_score(y, oof_final):9.6f} {ks_stat(oof_final, y):9.6f}")

ref_csv = pd.read_csv(V66 / "output/submission_v66_04_ext0.5_tm30_lgb_lean.csv")
ref = ref_csv.set_index("application_id").loc[test_ids].default_probability.to_numpy()
print(f"\nmax |rebuilt - submitted| : {np.abs(test_final - ref).max():.3e}")
print(f"Spearman vs submitted     : {spearmanr(test_final, ref).statistic:.10f}")
print(f"rank positions changed    : {int((rankdata(test_final) != rankdata(ref)).sum())}")

leg performance, out of fold
leg                AUC        KS
xgb_full      0.846756  0.525536


lgb_lean      0.846721  0.525311
lgb_full      0.846682  0.524974
fac_full      0.846662  0.524205
lgbx_full     0.846550  0.524089
cat_full      0.846311  0.525234
linear        0.842771  0.522712
CHAIN         0.846840  0.528713

max |rebuilt - submitted| : 1.110e-16
Spearman vs submitted     : 1.0000000000


rank positions changed    : 0


## 6. Why the leaderboard is now a measuring instrument

Eight scored submissions, split by whether they are the exact
`ext0.5 | tm30 | lgb_lean` chain:

| cluster | contents | n | mean | sd |
|---|---|---:|---:|---:|
| **A** | that exact chain | 3 | **0.56418** | **0.00007** |
| **B** | anything else | 5 | 0.56000 | 0.00123 |

Cluster A is `v66_04` 0.56424, `v70_07` 0.56419, `v68_04` 0.56411. Cluster B is
`core71` 0.56111, `mix80_mlp` 0.56073, `pair_mlp_v70_extra` 0.56045, `seq05`
0.55968, `v70_01` 0.55801.

**The +0.00418 gap is not luck.** The three cluster A members were trained on
different feature sets and different numbers of fold partitions — v66 on six
fold seeds, v68 on ten, `v70_07` on the v70 feature set entirely — and still
landed inside a 0.00013 band. The maximum of five draws from cluster B would
reach only 0.56142, so 0.56424 cannot be the luckiest draw of one distribution.

**And inside the chain, public has sd 0.00007.** That is the first precision
instrument this project has had. Any difference above 0.0002 is signal.

Two earlier readings are corrected by this. v74 concluded the public spread was
about 0.02 at a 3.000-row subset and that the leader's score was mostly noise;
v75 concluded 0.56424 was the maximum of the draws taken. Both were fitted to
cluster B behaviour, where they hold. Neither describes the chain, where the
score is reproducible to five decimals.

v75's calibration was nevertheless right where it was applied: it predicted
E[public] = 0.5596 for `mix80_mlp` and `pair_mlp_v70_extra` before they were
scored, and they came back 0.56073 and 0.56045.

**What is left to measure.** Leg quality does not move cluster A — `v70_07` and
`v68_04` prove that. So the remaining degrees of freedom are the chain's own
parameters, and exactly one value of each has ever reached public: `r` = 0.5,
`tm` = 0.30, core weight 2:1, reference leg `lgb_lean`. v61 and v66 did sweep
`r` and `tm`, but selected on OOF AUC, which v74 and v75 both measured as not
tracking public. On the metric that decides the competition this axis is
unmeasured.

Cluster C fixes one direction in advance. `qc_00` (`ext1.5`, reference
`fac_full`) scored 0.56171, `v58ext` 0.55900, `v66_02` 0.55877 — changing the
reference leg away from `lgb_lean` is expensive, so the sweep holds it fixed.

**The first submission is a test, not a bid.** The smallest perturbation
available inside the chain is the core weight, 2:1 to 3:1, at rho 0.999986
against the anchor. For comparison `seq05` sat at rho 0.99950 and still fell
0.00456. If the control holds near 0.5641 the surface is smooth and the sweep
is worth continuing; if it collapses to about 0.560, 0.56424 is a brittle
local maximum and the remaining submissions are saved.

In [7]:
d = json.loads((V76 / "output/summary_v76.json").read_text())

print("public ledger by cluster")
led = pd.DataFrame(d["ledger"])
for cl in ("A", "B", "C"):
    sub = led[led.cluster == cl]
    st = d["cluster_stats"][cl]
    print(f"\n  cluster {cl}: n={st['n']} mean={st['mean']:.5f} sd={st['sd']:.5f}")
    print(sub[["public", "name"]].to_string(index=False))
print(f"\ngap A - B = {d['gap_A_minus_B']:+.5f}")
print(f"instrument sd inside the chain = {d['instrument_sd']:.5f}")

print("\n\nthe sweep, one parameter changed per candidate")
t = pd.DataFrame(d["candidates"])[
    ["submit_order", "name", "oof_auc", "d_oof_ks", "spearman_vs_anchor",
     "rank_changes", "file"]]
print(t.sort_values("submit_order").to_string(index=False))

print("\nout-of-fold numbers above are context, not the selection key: v74 and")
print("v75 measured that they do not track public. The instrument is public.")
g = d["go_no_go"]
print(f"\ngo / no-go after submission 1 ({d['control']}):")
print(f"  >= {g['continue_at_or_above']}  smooth surface -> continue the sweep in order")
print(f"  <= {g['stop_at_or_below']}  brittle maximum -> stop, freeze the anchor")
print(f"  between      inconclusive -> send r0.75 only, then decide")

public ledger by cluster

  cluster A: n=3 mean=0.56418 sd=0.00007
 public                            name
0.56424    v66_04  ext0.5|tm30|lgb_lean
0.56419 v70_07  ext0.5|tm30v70|lgb_lean
0.56411    v68_04  ext0.5|tm30|lgb_lean

  cluster B: n=5 mean=0.56000 sd=0.00123
 public                        name
0.56111    v71     core71 MLP-heavy
0.56073           v75_03  mix80_mlp
0.56045  v75_04  pair_mlp_v70_extra
0.55968 v72     seq05 anchor+5% seq
0.55801   v70_01  raw leg v70_extra

  cluster C: n=4 mean=0.56065 sd=0.00212
 public                                   name
0.56312 v61_alt ext0.5|tm30|lean (weaker legs)
0.56171                qc_00   ext1.5|fac_full
0.55900                     v58ext  ext1.5|fac
0.55877          v66_02  plateau|tm20|fac_full

gap A - B = +0.00418
instrument sd inside the chain = 0.00007


the sweep, one parameter changed per candidate
 submit_order     name  oof_auc  d_oof_ks  spearman_vs_anchor  rank_changes                           file
            0   anc

## 7. Submission validation

Every candidate file must have 30.000 rows, the exact test id set, unique ids,
and finite probabilities in [0, 1].

In [8]:
test_id_set = set(pd.read_csv(DATA / "test_applications.csv",
                              usecols=["application_id"]).application_id)
files = sorted((V76 / "output").glob("submission_v76_*.csv"))
assert files, "no v76 submission files found"

report = []
for p in files:
    sub = pd.read_csv(p)
    checks = {
        "cols": list(sub.columns) == ["application_id", "default_probability"],
        "rows": len(sub) == 30_000,
        "unique": bool(sub.application_id.is_unique),
        "ids": set(sub.application_id) == test_id_set,
        "no_na": not bool(sub.isna().any().any()),
        "finite": bool(np.isfinite(sub.default_probability).all()),
        "range": bool(sub.default_probability.between(0, 1).all())}
    report.append({"file": p.name, **checks,
                   "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16]})
    assert all(checks.values()), f"{p.name} failed validation"

print(pd.DataFrame(report).to_string(index=False))

base = pd.read_csv(V76 / "output/submission_v76_00_anchor.csv")
base = base.set_index("application_id").loc[test_ids].default_probability.to_numpy()
print(f"\nv76_00_anchor vs the submitted v66_04 file: "
      f"Spearman {spearmanr(base, ref).statistic:.10f}, "
      f"rank changes {int((rankdata(base) != rankdata(ref)).sum())}")
print(f"\n{len(files)} files validated")
print("STATUS: PASS")

                          file  cols  rows  unique  ids  no_na  finite  range           sha256
  submission_v76_00_anchor.csv  True  True    True True   True    True   True acfb8ea882a92a41
submission_v76_01_core3to1.csv  True  True    True True   True    True   True ebfefc858ab4e2a5
   submission_v76_02_r0.75.csv  True  True    True True   True    True   True 35260db3730a3eec
   submission_v76_03_r0.25.csv  True  True    True True   True    True   True ed4e70d3d72c4b4b
  submission_v76_04_tm0.40.csv  True  True    True True   True    True   True 5c2e8fa78dcd21c8
  submission_v76_05_tm0.20.csv  True  True    True True   True    True   True 5d9fc9595505af49
   submission_v76_06_r1.00.csv  True  True    True True   True    True   True 56bbe88dda219f87
submission_v76_07_core1to1.csv  True  True    True True   True    True   True 3429f1312f8399da

v76_00_anchor vs the submitted v66_04 file: Spearman 1.0000000000, rank changes 0

8 files validated
STATUS: PASS


## 8. Recommendation and limits

**Final pick stays the anchor.** `agas/v66/output/submission_v66_04_ext0.5_tm30_lgb_lean.csv`,
public 0.56424 measured three times across different feature sets. Change the
final pick only if a sweep candidate comes back above it.

**Submit order.** `submission_v76_01_core3to1.csv` first, because it is the test
that decides whether the rest is worth sending. Then `r0.75` and `r0.25`, which
bracket the measured point so the gradient and curvature read off directly.
Then the `tm` pair. `r1.00` only if `r0.75` moved up.

**Do not submit** `submission_v76_00_anchor.csv`: it is rank-identical to the
anchor and returns 0.56424, wasting a slot. It exists as the rebuild check in
section 7.

**Limits.**

* The submitted vector is a rank, not a calibrated probability. KS is invariant
  under any increasing transform, so the ranking is the whole model, but the
  values must not be read as default rates.
* The instrument sd of 0.00007 rests on three observations of one recipe. It
  describes reproducibility of that recipe across feature sets and seeds; it is
  not a general claim about the leaderboard, and cluster B's sd is 0.00123.
* A sweep step is a different recipe, not a retrained version of the same one,
  so its deviation mixes the parameter effect with whatever fragility the
  surface has. `seq05` shows fragility is real: rho 0.99950 cost 0.00456. This
  is why the control is sent first.
* `customer_profiles` has no as-of date, so profile fields are used as given.
  Every other source is truncated at the decision time, re-verified in
  section 2 on every run.
* `features.py` does not apply a positive `days_ago` filter in three of its main
  blocks. That is safe only because the provided tables carry no post-cutoff
  records, which section 2 re-checks.
* No external data, no AutoML, no automated feature engineering. v76 trains
  nothing; it re-assembles stored leg predictions.